# Taller B5-T1 · Generación de datos financieros sintéticos
## Notebook 04 — La malla real × sintético

**Esta es la pregunta del taller.** Los notebooks anteriores construyeron las piezas: el
dataset con splits purgados (01), la arquitectura downstream congelada y los baselines
clásicos (02), y seis generadores auditados contra los hechos estilizados (03). Aquí se
responde: **¿cuántos datos sintéticos ayudan, en qué régimen de escasez de datos reales, y
qué generador los produce mejor?**

### El experimento

Para cada combinación (N_real, generador, ratio, semilla):

1. Se submuestrean `N_real` ventanas del train.
2. El generador se **reentrena con esas `N_real` ventanas y solo esas**.
3. Se generan `N_synth = ratio × N_real` ventanas sintéticas.
4. Se entrena la arquitectura **congelada** del notebook 02 sobre la mezcla.
5. Se evalúa en el **test real**, intocado y siempre el mismo.

### Las cuatro reglas que hacen esto interpretable

| Regla | Por qué |
|---|---|
| **El generador se reentrena en cada celda** | Ajustarlo con las 102k ventanas y luego *simular* escasez sería fuga: los sintéticos llevarían información de datos que el escenario declara no tener, y la curva saldría optimista. |
| **Submuestreo por fechas, no por filas** | En una fecha dada todas las ventanas del panel cuentan la misma historia de mercado. Un muestreo de filas daría un N nominal muy superior al efectivo. |
| **Validación siempre real** | Si el early-stopping mirase datos sintéticos, optimizaríamos la distribución equivocada. |
| **Arquitectura e hiperparámetros congelados** | Si una celda rinde distinto, la única explicación posible son los datos. |

> **Métrica principal: `delta_r2`** — la ganancia de R² en test de cada celda frente a la
> celda de *solo real* con el mismo N. Responde literalmente al enunciado: para el mismo
> presupuesto de datos reales, ¿cuánto añade el sintético? El umbral de relevancia lo fijó
> el notebook 02: **±0,006**, la dispersión entre semillas del modelo con datos reales.
> Cualquier efecto por debajo de eso es ruido de optimización, no señal.

## 0 · Setup y configuración de la malla

**Dos tamaños de malla.** `REDUCIDA` valida el pipeline y ya cubre el régimen de escasez,
que es donde está el hallazgo; `COMPLETA` añade el extremo de datos abundantes y el ratio
10×. La diferencia de coste es de ~6×, y sale casi entera de dos celdas: `N=10.000` con
`ratio=10` entrena sobre 110.000 ventanas, tanto como el dataset completo.

**Submuestreo de la validación.** La validación real tiene 17.146 ventanas y se evalúa en
cada época de cada celda: a N pequeño ese coste fijo **domina el tiempo total** de la malla
(medido: ~45 s por celda con N=300). Como su único papel es elegir la época de la que se
conservan los pesos, 5.000 ventanas reales bastan de sobra y el coste por celda cae unas
tres veces. Es una decisión de eficiencia que no toca ninguna comparación: la validación es
idéntica en todas las celdas.

In [ ]:
import sys, json, time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.config import Config, set_global_seed
from src.eda import PALETTE, _style
from src.malla import (plan_de_malla, ejecutar_malla, resumir, delta_vs_solo_real, SOLO_REAL)
from src.training import get_device

cfg = Config()
set_global_seed(cfg.seed)
device = get_device()
print("Device:", device)

# ---- Tamaño de la malla -------------------------------------------------
MALLA = "REDUCIDA"          # "REDUCIDA" o "COMPLETA"

CONFIG_MALLA = {
    "REDUCIDA": dict(n_reales=[250, 1000, 3000], ratios=[0, 1, 3], seeds=[0, 1, 2]),
    "COMPLETA": dict(n_reales=[250, 1000, 3000, 10000], ratios=[0, 1, 3, 10], seeds=[0, 1, 2]),
}[MALLA]

GENERADORES = ["jitter", "gaussiana", "block_bootstrap", "vae", "wgan_gp", "realnvp"]
N_VAL = 5000                # submuestra de validación (real) — ver nota arriba
RUTA_CSV = cfg.out_dir / f"malla_{MALLA.lower()}.csv"

print(f"Malla {MALLA}: {CONFIG_MALLA}")
print(f"Resultados -> {RUTA_CSV}")

In [ ]:
# ---- Datos: todo lo que la malla puede tocar ----------------------------
d   = np.load(cfg.out_dir / "windows_dataset.npz")
std = json.load(open(cfg.out_dir / "standardizer.json"))
ref = json.load(open(cfg.out_dir / "downstream_reference.json"))   # arquitectura CONGELADA
meta = pd.read_parquet(cfg.out_dir / "windows_meta.parquet")
meta_train = meta[meta.split == "train"].reset_index(drop=True)

S = lambda a, m, s: ((a - std[m]) / std[s]).astype(np.float32)

# validación real submuestreada, fija para TODAS las celdas
rng_val = np.random.default_rng(cfg.seed)
iv = rng_val.choice(len(d["X_val"]), min(N_VAL, len(d["X_val"])), replace=False)

CONTEXTO = dict(
    Xs_train=S(d["X_train"], "x_mu", "x_sd"), ys_train=S(d["y_train"], "y_mu", "y_sd"),
    meta_train=meta_train,
    Xs_val=S(d["X_val"], "x_mu", "x_sd")[iv], ys_val=S(d["y_val"], "y_mu", "y_sd")[iv],
    Xs_test=S(d["X_test"], "x_mu", "x_sd"), y_test_fisico=d["y_test"],
    std=std, ref=ref, device=device,
)

print(f"Arquitectura congelada: {ref['arch']} {ref['arch_kwargs']}")
print(f"Referencia con datos reales completos: R² test = "
      f"{np.mean([v['test_r2'] for v in ref['metricas_seeds'].values()]):.4f}")
print(f"train {meta_train.shape[0]:,} ventanas / {meta_train.date_t.nunique():,} fechas | "
      f"val {len(iv):,} (submuestra) | test {len(d['X_test']):,}")

## 1 · Ejecución de la malla

El plan se ordena de **coste creciente**: si la ejecución se interrumpe, lo ya calculado es
el régimen de escasez, que es el que más importa. Cada celda se escribe al CSV en cuanto
termina, así que **relanzar esta celda reanuda donde se quedó** sin recalcular nada.

In [ ]:
plan = plan_de_malla(generadores=GENERADORES, **CONFIG_MALLA)
print(f"{len(plan)} celdas en el plan "
      f"({len(CONFIG_MALLA['n_reales'])} niveles de N × "
      f"{len(CONFIG_MALLA['ratios'])} ratios × {len(GENERADORES)} generadores × "
      f"{len(CONFIG_MALLA['seeds'])} semillas)")

In [ ]:
t0 = time.time()
df = ejecutar_malla(plan, RUTA_CSV, **CONTEXTO)
print(f"\nMalla terminada en {(time.time() - t0)/60:.1f} min | {len(df)} celdas en el CSV")

## 2 · Resultados

### 2.1 Tabla resumen

In [ ]:
resumen = resumir(df)
delta = delta_vs_solo_real(resumen)

# referencia de cada N: cuánto rinde el modelo con SOLO datos reales
print("Referencia — solo datos reales:")
display(resumen[resumen.generador == SOLO_REAL]
        [["n_real", "test_r2_media", "test_r2_sd"]].round(4).to_string(index=False))

# tabla principal: ganancia frente a esa referencia
tabla = delta.pivot_table(index=["n_real", "generador"], columns="ratio", values="delta_r2")
tabla.round(4)

### 2.2 La figura del taller: ganancia frente a proporción de sintéticos

Un panel por nivel de escasez. La banda gris es el **umbral de relevancia** (±0,006, la
dispersión entre semillas del notebook 02): cualquier curva dentro de esa banda es ruido,
no efecto.

In [ ]:
n_paneles = len(CONFIG_MALLA["n_reales"])
fig, axes = plt.subplots(1, n_paneles, figsize=(4.2 * n_paneles, 3.8), sharey=True)
axes = np.atleast_1d(axes)
colores = [PALETTE[c] for c in ("blue", "orange", "green", "vermillion", "purple", "sky")]
UMBRAL = 0.006

for ax, n in zip(axes, CONFIG_MALLA["n_reales"]):
    sub = delta[delta.n_real == n]
    ax.axhspan(-UMBRAL, UMBRAL, color=PALETTE["grey"], alpha=0.18, zorder=0)
    ax.axhline(0, color=PALETTE["grey"], lw=0.9)
    for col, g in zip(colores, GENERADORES):
        s = sub[sub.generador == g].sort_values("ratio")
        if len(s):
            ax.plot(s.ratio, s.delta_r2, "o-", color=col, lw=1.5, ms=5, label=g)
    ax.set_title(f"N_real = {n:,}", fontsize=10)
    ax.set_xlabel("ratio  sintéticos / reales")
    _style(ax)
axes[0].set_ylabel("Δ R² frente a solo real")
axes[-1].legend(fontsize=7, loc="best")
fig.suptitle("¿Cuánto aporta el dato sintético, y en qué régimen de escasez?")
fig.tight_layout()
fig.savefig(cfg.fig_dir / "14_delta_r2_por_escasez.png", dpi=120, bbox_inches="tight")

### 2.3 Curvas de aprendizaje: R² frente a datos reales disponibles

La lectura complementaria — y la que reproduce la forma de la gráfica que el profesor
mostró en clase. La línea negra discontinua es el techo: el modelo entrenado con las
**102.406 ventanas reales completas** (notebook 02).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
techo = np.mean([v["test_r2"] for v in ref["metricas_seeds"].values()])

base = resumen[resumen.generador == SOLO_REAL].sort_values("n_real")
ax.plot(base.n_real, base.test_r2_media, "o--", color="black", lw=2, ms=7,
        label="solo real", zorder=5)

RATIO_FOCO = 3 if 3 in CONFIG_MALLA["ratios"] else max(CONFIG_MALLA["ratios"])
for col, g in zip(colores, GENERADORES):
    s = resumen[(resumen.generador == g) & (resumen.ratio == RATIO_FOCO)].sort_values("n_real")
    if len(s):
        ax.plot(s.n_real, s.test_r2_media, "o-", color=col, lw=1.5, ms=5, label=g)

ax.axhline(techo, color=PALETTE["grey"], ls=":", lw=1.5)
ax.text(ax.get_xlim()[0], techo, f" techo: 102k reales (R²={techo:.3f})",
        va="bottom", fontsize=8, color=PALETTE["grey"])
ax.set_xscale("log")
ax.set_xlabel("ventanas reales disponibles (escala log)")
ax.set_ylabel("R² en test")
ax.set_title(f"Curva de aprendizaje — sintéticos añadidos con ratio {RATIO_FOCO}×")
ax.legend(fontsize=8)
_style(ax)
fig.tight_layout()
fig.savefig(cfg.fig_dir / "15_curva_aprendizaje.png", dpi=120, bbox_inches="tight")

### 2.4 ¿Predice la auditoría del notebook 03 el efecto downstream?

La pregunta que cierra el círculo del taller. En el notebook 03 medimos la fidelidad de
cada generador (discriminative AUC, curtosis, clustering, `err_corr_xy`) **sin mirar el
modelo downstream**. Si esas métricas anticipan la ganancia real, la auditoría sirve para
elegir generador sin tener que barrer una malla entera. Si no, no sirve — y eso también hay
que decirlo.

In [ ]:
# Métricas de fidelidad medidas en el notebook 03 (sobre las 102k de train)
AUDITORIA_NB03 = pd.DataFrame({
    "generador":          ["jitter", "gaussiana", "block_bootstrap", "vae", "wgan_gp", "realnvp"],
    "discriminative_auc": [0.507,    0.958,       0.760,             0.899,  0.809,     0.652],
    "curtosis_x":         [23.565,   0.031,       26.161,            1.620,  5.101,     13.910],
    "acf_abs_lag1":       [0.058,   -0.015,       0.125,            -0.007, -0.012,     0.046],
    "err_corr_xy":        [0.014,    0.013,       0.042,             0.027,  0.027,     0.025],
})

# Ganancia media por generador en el régimen de escasez (el N más pequeño)
n_min = min(CONFIG_MALLA["n_reales"])
ganancia = (delta[delta.n_real == n_min].groupby("generador")["delta_r2"]
            .mean().rename("delta_r2_medio").reset_index())

cruce = AUDITORIA_NB03.merge(ganancia, on="generador")
print(f"Fidelidad (nb03) frente a ganancia downstream con N_real = {n_min}:")
display(cruce.round(3))

corr = cruce[["discriminative_auc", "curtosis_x", "acf_abs_lag1", "err_corr_xy"]].corrwith(
    cruce["delta_r2_medio"], method="spearman")
print("\nCorrelación de Spearman de cada métrica de fidelidad con la ganancia downstream:")
print(corr.round(3).to_string())
print("\n(n = 6 generadores: estas correlaciones son indicativas, no concluyentes)")

## 3 · Análisis crítico

> Esta sección se completa **después** de ejecutar la malla. Lo que sigue son las preguntas
> concretas que las figuras y tablas anteriores deben responder, con las hipótesis que
> traíamos de los notebooks previos y los criterios para darlas por confirmadas o
> refutadas. Escribir las hipótesis *antes* de ver los resultados es lo que impide
> racionalizar a posteriori cualquier cosa que salga.

**H1 · El sintético solo ayuda en régimen de escasez.** Es la hipótesis central, heredada
del resultado del propio profesor (con 500 reales el sintético divide el error; con 20.000
no hace nada). *Criterio*: `delta_r2` claramente positivo y por encima de ±0,006 en el N
más pequeño, decreciendo hacia cero al aumentar N. *Si se refuta*: si el sintético no ayuda
ni siquiera con 250 ventanas reales, la conclusión del taller pasa a ser negativa — que es
un resultado perfectamente publicable, y más honesto que forzar una malla hasta encontrar
una celda favorable.

**H2 · Existe un óptimo intermedio de proporción.** El profesor lo dijo en clase: *"si me
paso, el modelo solo aprende de los sintéticos e ignora los buenos, que son los de
verdad"*. *Criterio*: la curva de `delta_r2` frente al ratio sube y luego baja, en lugar de
crecer monótonamente.

**H3 · El realismo no basta.** El notebook 03 dejó dos candidatos con perfiles opuestos: el
**block bootstrap** reproduce colas y clustering casi perfectamente pero es el que peor
preserva la relación X–y (`err_corr_xy` 0,042, el doble que cualquier otro); la
**gaussiana** falla todos los hechos estilizados pero preserva la correlación tan bien como
el mejor (0,013). *Criterio*: si la gaussiana bate al block bootstrap, el mecanismo por el
que el sintético ayuda es **regularización**, no realismo — y `err_corr_xy` es mejor
predictor que la curtosis o el AUC. Es el resultado que más me sorprendería y el que mejor
titular daría a la presentación.

**H4 · Las redes generativas pierden justo donde harían falta.** Con 250 ventanas en
dimensión 61, un WGAN-GP o un RealNVP tienen más parámetros que datos. *Criterio*: sus
`delta_r2` deberían ser los peores en el N más pequeño y mejorar al crecer N. Si se
confirma, la conclusión práctica es incómoda y valiosa: **los generadores neuronales
necesitan más datos de los que pretenden ahorrarte**, y en el régimen donde el dato
sintético tendría más valor son precisamente los peores. El RealNVP ya avisó de esto en el
notebook 03, perdiendo el 82 % de su ventaja de verosimilitud al salir de train.

**H5 · El jitter es difícil de batir.** No genera información nueva —sus muestras son
ventanas reales con ruido— pero es exactamente eso lo que lo convierte en una
regularización barata y sin riesgo de distorsionar la relación X–y. *Criterio*: si ningún
generador neuronal lo supera, el taller concluye que el aparato generativo no se paga a sí
mismo en este problema.

**Qué mirar además de la media.** La columna `test_r2_sd` del resumen: con N=250 la
varianza entre semillas puede ser grande, y una diferencia de medias que no supere esa
dispersión no es una diferencia. Y `epoca_mejor`: si en las celdas con mucho sintético el
modelo elige épocas muy tempranas, es señal de que el sintético empuja hacia una
distribución que la validación real castiga enseguida.

## 4 · Persistencia

El CSV de la malla es el artefacto que alimenta la presentación. Se guarda en
`data/processed/` junto al resto, y se versiona.

In [ ]:
resumen.to_csv(cfg.out_dir / f"malla_{MALLA.lower()}_resumen.csv", index=False)
delta.to_csv(cfg.out_dir / f"malla_{MALLA.lower()}_delta.csv", index=False)
print("Guardado:")
for p in sorted(cfg.out_dir.glob(f"malla_{MALLA.lower()}*.csv")):
    print(f"  {p.name}  ({p.stat().st_size/1024:.0f} KB)")